# Kokoro-82M：Google Colab T4 批量生成英语 WAV

这个 Notebook 使用轻量级 **Kokoro-82M** 替代 Qwen3-TTS，面向 Google Colab 的 Tesla T4。

功能：
- 检查 CUDA GPU
- 使用 Kokoro-82M 美式英语模型
- 先生成 10 条英语句子
- 每条句子保存为独立 WAV
- 保存到 Google Drive
- 已生成文件自动跳过
- 普通错误自动重试
- 保存 `progress.json` 和 `manifest.csv`
- 在 Colab 中直接试听
- 可打包 ZIP 下载

运行前请选择：**运行时 → 更改运行时类型 → T4 GPU**。

In [ ]:
# 1. 检查 GPU
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        '没有检测到 CUDA GPU。请在 Colab 中选择：运行时 → 更改运行时类型 → T4 GPU。'
    )

GPU_NAME = torch.cuda.get_device_name(0)
GPU_CAPABILITY = torch.cuda.get_device_capability(0)

print('PyTorch:', torch.__version__)
print('GPU:', GPU_NAME)
print('Compute Capability:', GPU_CAPABILITY)
print('CUDA:', torch.version.cuda)

if 'T4' not in GPU_NAME.upper():
    print('提示：当前不是 Tesla T4，但只要是 CUDA GPU，通常也可以运行。')

subprocess.run(['nvidia-smi'], check=False)


In [ ]:
# 2. 安装 Kokoro 和英语音素依赖
%pip install -q -U "kokoro>=0.9.4" soundfile
!apt-get -qq -y install espeak-ng > /dev/null 2>&1

print('依赖安装完成。')


In [ ]:
# 3. 挂载 Google Drive
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

OUTPUT_DIR = Path('/content/drive/MyDrive/kokoro_tts_test_10')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('输出目录:', OUTPUT_DIR)


In [ ]:
# 4. 准备 10 条英语句子
# 后续可直接替换为自己的英语句子。

sentences = [
    'Good morning. I hope you have a wonderful day.',
    'The weather is beautiful, so let us take a walk outside.',
    'Please read each sentence clearly and naturally.',
    'Learning English takes patience, practice, and confidence.',
    'She arrived at the station five minutes before the train left.',
    'Could you tell me where the nearest library is?',
    'Technology has changed the way people work and communicate.',
    'He paused for a moment before answering the difficult question.',
    'Every small improvement brings us closer to our goal.',
    'Thank you for listening, and I will see you next time.'
]

records = [
    {'id': index + 1, 'text': text.strip()}
    for index, text in enumerate(sentences)
    if text.strip()
]

print('句子数量:', len(records))
for item in records:
    print(f"{item['id']:05d}: {item['text']}")


In [ ]:
# 5. 加载 Kokoro-82M 到 T4
from kokoro import KPipeline

# lang_code='a'：美式英语
# lang_code='b'：英式英语
LANG_CODE = 'a'

# 推荐美式英语女声：
# af_heart：官方评级较高，默认推荐
# af_bella：另一种高质量女声
# am_fenrir / am_michael：男声
VOICE = 'af_heart'

# 1.0 为正常速度；0.9 更慢；1.1 更快
SPEED = 1.0
SAMPLE_RATE = 24000

print('正在加载 Kokoro-82M...')
pipeline = KPipeline(
    lang_code=LANG_CODE,
    repo_id='hexgrad/Kokoro-82M',
    device='cuda',
)

print('模型加载成功。')
print('GPU:', GPU_NAME)
print('语言:', LANG_CODE)
print('音色:', VOICE)
print('语速:', SPEED)


In [ ]:
# 6. 生成 WAV：断点续跑、错误重试、已完成跳过

import csv
import gc
import json
import time
import traceback
import numpy as np
import soundfile as sf
import torch

MAX_RETRIES = 3
CHUNK_GAP_SECONDS = 0.08
PROGRESS_FILE = OUTPUT_DIR / 'progress.json'
MANIFEST_FILE = OUTPUT_DIR / 'manifest.csv'

def output_path(item):
    return OUTPUT_DIR / f"{item['id']:05d}.wav"

def audio_to_numpy(audio):
    if torch.is_tensor(audio):
        audio = audio.detach().float().cpu().numpy()
    audio = np.asarray(audio, dtype=np.float32).squeeze()
    if audio.ndim != 1:
        raise ValueError(f'音频维度异常: {audio.shape}')
    if not np.isfinite(audio).all():
        raise ValueError('生成音频包含 NaN 或 Inf。')
    return audio

def synthesize_one(text):
    parts = []

    # Kokoro 会在文本较长时自动分段，因此需要把分段音频拼接回一个文件。
    generator = pipeline(
        text,
        voice=VOICE,
        speed=SPEED,
        split_pattern=r'\n+',
    )

    for _, _, audio in generator:
        if audio is not None:
            part = audio_to_numpy(audio)
            if part.size > 0:
                parts.append(part)

    if not parts:
        raise RuntimeError('模型没有返回有效音频。')

    if len(parts) == 1:
        return parts[0]

    silence = np.zeros(
        int(SAMPLE_RATE * CHUNK_GAP_SECONDS),
        dtype=np.float32,
    )

    combined = []
    for index, part in enumerate(parts):
        if index > 0:
            combined.append(silence)
        combined.append(part)

    return np.concatenate(combined)

def save_progress(completed_ids, failures):
    payload = {
        'model': 'hexgrad/Kokoro-82M',
        'gpu': GPU_NAME,
        'language_code': LANG_CODE,
        'voice': VOICE,
        'speed': SPEED,
        'sample_rate': SAMPLE_RATE,
        'completed_ids': sorted(completed_ids),
        'failures': failures,
        'updated_at': time.strftime('%Y-%m-%d %H:%M:%S'),
    }
    PROGRESS_FILE.write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )

completed_ids = {
    item['id'] for item in records if output_path(item).exists()
}
failures = {}
pending = [item for item in records if item['id'] not in completed_ids]

print(f'已完成 {len(completed_ids)} 条，待生成 {len(pending)} 条。')

for item in pending:
    last_error = None

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            print(
                f"生成 [{item['id']}]，第 {attempt}/{MAX_RETRIES} 次尝试："
                f"{item['text']}"
            )

            audio = synthesize_one(item['text'])
            target = output_path(item)
            sf.write(str(target), audio, SAMPLE_RATE, subtype='PCM_16')

            completed_ids.add(item['id'])
            failures.pop(str(item['id']), None)
            save_progress(completed_ids, failures)

            duration = len(audio) / SAMPLE_RATE
            print(f'已保存: {target.name}，时长 {duration:.2f} 秒')
            last_error = None
            break

        except Exception as exc:
            last_error = exc
            print(f'生成失败: {type(exc).__name__}: {exc}')
            print(traceback.format_exc(limit=1))
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            time.sleep(attempt * 2)

    if last_error is not None:
        failures[str(item['id'])] = {
            'text': item['text'],
            'error': f'{type(last_error).__name__}: {last_error}',
        }
        save_progress(completed_ids, failures)
        print(f"第 {item['id']} 条最终失败，已记录，继续下一条。")

# 保存总清单
with MANIFEST_FILE.open('w', newline='', encoding='utf-8-sig') as file:
    writer = csv.DictWriter(
        file,
        fieldnames=['id', 'filename', 'text', 'status'],
    )
    writer.writeheader()
    for item in records:
        writer.writerow({
            'id': item['id'],
            'filename': output_path(item).name,
            'text': item['text'],
            'status': 'completed' if output_path(item).exists() else 'failed',
        })

print('\n生成结束。')
print('成功:', len(completed_ids))
print('失败:', len(failures))
print('目录:', OUTPUT_DIR)


In [ ]:
# 7. 在 Colab 中试听 WAV
from IPython.display import Audio, Markdown, display

wav_files = sorted(OUTPUT_DIR.glob('*.wav'))
print('找到 WAV 文件:', len(wav_files))

for wav_file in wav_files[:10]:
    display(Markdown(f'**{wav_file.name}**'))
    display(Audio(filename=str(wav_file)))


In [ ]:
# 8. 可选：打包 ZIP 下载
import shutil
from IPython.display import FileLink, display

ZIP_BASE = '/content/kokoro_tts_test_10'
zip_path = shutil.make_archive(
    ZIP_BASE,
    'zip',
    root_dir=str(OUTPUT_DIR),
)

print('ZIP:', zip_path)
display(FileLink(zip_path))


## 修改为自己的句子

修改第 4 格中的 `sentences` 即可。

## 修改音色

第 5 格中可修改：

```python
VOICE = 'af_heart'
```

推荐尝试：
- `af_heart`：美式英语女声，默认推荐
- `af_bella`：美式英语女声
- `am_fenrir`：美式英语男声
- `am_michael`：美式英语男声
- `bf_emma`：英式英语女声，需要同时设置 `LANG_CODE = 'b'`

## 断点续跑

重新运行第 6 格时，已经存在的 WAV 会自动跳过。

更换句子或音色后，如需全部重新生成，请删除 Google Drive 中旧的 WAV 文件，或修改 `OUTPUT_DIR`。